# FlightInsight — Data Exploration

Live flight data from OpenSky Network. No server needed — this notebook calls the API directly.

**Run with:** `uv run jupyter notebook` from the project root.

**Sections:**
1. Setup & fetch live data
2. Basic stats
3. Top countries
4. Altitude distribution
5. Speed distribution
6. Live map
7. Vertical rate (climb / level / descend)
8. Eval questions

## 1. Setup & fetch live data

In [ ]:
import os
import sys

import httpx
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

OPENSKY_CLIENT_ID     = os.getenv("OPENSKY_CLIENT_ID", "")
OPENSKY_CLIENT_SECRET = os.getenv("OPENSKY_CLIENT_SECRET", "")

BBOX_EUROPE = dict(lamin=35.0, lomin=-10.0, lamax=70.0, lomax=40.0)

print(f"Python {sys.version}")
print(f"Authenticated: {bool(OPENSKY_CLIENT_ID)}")

In [ ]:
COLUMNS = [
    "icao24", "callsign", "origin_country", "time_position",
    "last_contact", "longitude", "latitude", "baro_altitude",
    "on_ground", "velocity", "true_track", "vertical_rate",
    "sensors", "geo_altitude", "squawk", "spi", "position_source",
]


def get_token() -> str | None:
    if not OPENSKY_CLIENT_ID:
        return None
    r = httpx.post(
        "https://auth.opensky-network.org/auth/realms/opensky-network/protocol/openid-connect/token",
        data={
            "grant_type": "client_credentials",
            "client_id": OPENSKY_CLIENT_ID,
            "client_secret": OPENSKY_CLIENT_SECRET,
        },
        timeout=10,
    )
    r.raise_for_status()
    return r.json()["access_token"]


def fetch_states(bbox: dict) -> pd.DataFrame:
    token = get_token()
    headers = {"Authorization": f"Bearer {token}"} if token else {}
    r = httpx.get(
        "https://opensky-network.org/api/states/all",
        params=bbox,
        headers=headers,
        timeout=15,
    )
    r.raise_for_status()
    states = r.json().get("states") or []
    if not states:
        return pd.DataFrame(columns=COLUMNS)
    ncols = len(states[0])
    df = pd.DataFrame(states, columns=COLUMNS[:ncols])
    df["callsign"] = df["callsign"].str.strip()
    return df


print("Fetching live flights over Europe...")
df = fetch_states(BBOX_EUROPE)
print(f"Got {len(df)} aircraft")
df.head()

## 2. Basic stats

In [ ]:
airborne  = df[df["on_ground"] == False].copy()
grounded  = df[df["on_ground"] == True].copy()

avg_alt   = airborne["baro_altitude"].mean()
avg_spd   = airborne["velocity"].mean()
max_spd   = airborne["velocity"].max()

print(f"Total aircraft tracked : {len(df)}")
print(f"Airborne               : {len(airborne)}  ({len(airborne)/len(df)*100:.1f}%)")
print(f"On ground              : {len(grounded)}")
print(f"Countries represented  : {df['origin_country'].nunique()}")
print(f"Avg altitude           : {avg_alt:.0f} m  ({avg_alt/0.3048/100:.0f} FL)")
print(f"Avg speed              : {avg_spd:.1f} m/s  =  {avg_spd*3.6:.0f} km/h")
print(f"Max speed              : {max_spd:.1f} m/s  =  {max_spd*3.6:.0f} km/h")

## 3. Top countries

In [ ]:
# pandas 2.x: value_counts().reset_index() gives columns [original_name, "count"]
top_countries = (
    df["origin_country"]
    .value_counts()
    .head(15)
    .reset_index()
)
# columns are: "origin_country", "count"

fig = px.bar(
    top_countries,
    x="origin_country",
    y="count",
    title="Top 15 countries by number of aircraft (Europe, live)",
    labels={"origin_country": "Country", "count": "Aircraft"},
    color="count",
    color_continuous_scale="Blues",
)
fig.update_layout(coloraxis_showscale=False)
fig.show()

## 4. Altitude distribution

In [ ]:
alt = airborne["baro_altitude"].dropna()

fig = px.histogram(
    x=alt,
    nbins=50,
    title="Altitude distribution — airborne aircraft (meters)",
    labels={"x": "Altitude (m)", "y": "Aircraft"},
    color_discrete_sequence=["#1f77b4"],
)
fig.add_vline(
    x=alt.mean(),
    line_dash="dash",
    annotation_text=f"avg {alt.mean():.0f} m",
)
fig.show()

cruising = airborne[(airborne["baro_altitude"] >= 9000) & (airborne["baro_altitude"] <= 12500)]
print(f"At cruising altitude (9–12.5 km): {len(cruising)} aircraft ({len(cruising)/len(airborne)*100:.1f}% of airborne)")

## 5. Speed distribution

In [ ]:
spd_kmh = (airborne["velocity"].dropna() * 3.6)

fig = px.histogram(
    x=spd_kmh,
    nbins=60,
    title="Speed distribution — airborne aircraft (km/h)",
    labels={"x": "Speed (km/h)", "y": "Aircraft"},
    color_discrete_sequence=["#ff7f0e"],
)
fig.add_vline(
    x=spd_kmh.mean(),
    line_dash="dash",
    annotation_text=f"avg {spd_kmh.mean():.0f} km/h",
)
fig.show()

print(spd_kmh.describe(percentiles=[0.1, 0.5, 0.9]).round(0).to_string())

## 6. Live map — all airborne aircraft

In [ ]:
map_df = airborne[["icao24", "callsign", "origin_country", "latitude", "longitude", "baro_altitude", "velocity"]].dropna(subset=["latitude", "longitude"]).copy()
map_df["speed_kmh"]   = (map_df["velocity"] * 3.6).round(0)
map_df["altitude_km"] = (map_df["baro_altitude"] / 1000).round(2)

fig = px.scatter_mapbox(
    map_df,
    lat="latitude",
    lon="longitude",
    color="altitude_km",
    hover_name="callsign",
    hover_data={"origin_country": True, "speed_kmh": True, "altitude_km": True, "latitude": False, "longitude": False},
    color_continuous_scale="Viridis",
    zoom=3,
    center={"lat": 50, "lon": 10},
    mapbox_style="open-street-map",
    title=f"Live flights over Europe ({len(map_df)} aircraft)",
    labels={"altitude_km": "Altitude (km)"},
)
fig.update_traces(marker_size=5)
fig.show()

## 7. Vertical rate — climb / level / descend

In [ ]:
vr = airborne["vertical_rate"].dropna()

climbing   = int((vr >  1).sum())
descending = int((vr < -1).sum())
level      = int(((vr >= -1) & (vr <= 1)).sum())

fig = px.pie(
    names=["Climbing", "Level", "Descending"],
    values=[climbing, level, descending],
    title="Flight phase (airborne)",
    color_discrete_sequence=["#2ecc71", "#3498db", "#e74c3c"],
)
fig.show()
print(f"Climbing: {climbing} | Level: {level} | Descending: {descending}")

## 8. Three "waouh" eval questions

Questions the final FlightInsight agent must answer perfectly. These anchor the Day 12 eval dataset.

---

**Q1 — Real-time + ranking (REALTIME)**
> *"Which 5 aircraft are currently flying the fastest over Europe, and what are their callsigns and speeds?"*

Impressive because: needs live data, sorting, and clean formatting. The speed numbers (850+ km/h) surprise people.

---

**Q2 — Regulation + specifics (KNOWLEDGE)**
> *"My flight from Paris to New York was cancelled 10 days before departure. Under EU 261/2004, what compensation am I entitled to and what are the airline's obligations?"*

Impressive because: the agent must hit the 14-day threshold, €600 for long-haul, and re-routing obligation — all from the RAG corpus.

---

**Q3 — Hybrid: live data + knowledge (HYBRID)**
> *"Are there currently any flights showing unusual altitude or speed patterns over France? What could explain them?"*

Impressive because: combines the anomaly detection model, live OpenSky data, and aviation knowledge to give a reasoned expert answer.